# Visium HD - SCA003

In [1]:
# TODO: Visium HD reader in harpy currently does not work due to bug fix in spatialdata-io (https://github.com/scverse/spatialdata-io/pull/303). Update harpy reader.
# TODO: Discuss option with Arne to overwrite tissue positions with our own registered coordinates.
# TODO: Rebin data based on table layer
# TODO: BMK reader, Nova-ST reader, Curio reader, StereoSeq reader

## Import packages

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import harpy as hp
import spatialdata as sd
import pandas as pd
import numpy as np
import dask as da
import os
import gc
import geopandas as gpd
import matplotlib
from matplotlib import pyplot as plt
import seaborn as sns
import h5py
import json
import gc

from dask_image.imread import imread
import spatialdata_plot

from spatialdata.transformations import get_transformation, remove_transformation, set_transformation

c:\Users\julien.mortier\conda-envs\harpy\lib\site-packages\xarray_schema\__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


In [4]:
import sys
sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath(''))))
from utils.in_tissue import add_in_tissue, assign_region
from utils.plotting import best_datatree_scale, rel_fontsize
from utils.qc import qc_metrics
from _visium_hd import visium_hd

## Create sdata

In [5]:
dataset_id = "SCA003"

path_outs = f'D:/2024-07-UTBenchmark-SpC/data/processed/Visium_HD/{dataset_id}/SpaceRanger/outs'
path_img = f'D:/2024-07-UTBenchmark-SpC/data/raw/Visium_HD/2/IMAGES/{dataset_id}_HE_hires.tif'
output_path = f'D:/2024-07-UTBenchmark-SpC/data/processed/Visium_HD/{dataset_id}/harpy/sdata.zarr'

pipeline_name = 'SpaceRanger'
species = 'mouse'
tissue_annotation_file = f'D:/2024-07-UTBenchmark-SpC/data/processed/Visium_HD/{dataset_id}/tissue_annotation/tissue.geojson'

In [6]:
sdata = sd.read_zarr(output_path)

version mismatch: detected: RasterFormatV02, requested: FormatV04
c:\Users\julien.mortier\conda-envs\harpy\lib\site-packages\zarr\creation.py:614: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)
version mismatch: detected: RasterFormatV02, requested: FormatV04
c:\Users\julien.mortier\conda-envs\harpy\lib\site-packages\zarr\creation.py:614: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)
c:\Users\julien.mortier\conda-envs\harpy\lib\site-packages\zarr\creation.py:614: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)
c:\Users\julien.mortier\conda-envs\harpy\lib\site-packages\zarr\creation.py:614: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)
c:\Users\julien.mortier\conda-envs\harpy\lib

In [6]:
sdata = visium_hd(
    path=path_outs, 
    dataset_id=dataset_id,
    filtered_counts_file=False,
    bin_size=[2, 8, 16, 32], 
    bins_as_squares=True, 
    fullres_image_file=path_img,
    load_all_images=True,
    output=output_path
)
sdata

c:\Users\julien.mortier\conda-envs\harpy\lib\site-packages\anndata\_core\anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
c:\Users\julien.mortier\conda-envs\harpy\lib\site-packages\anndata\_core\anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
c:\Users\julien.mortier\conda-envs\harpy\lib\site-packages\anndata\_core\anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
c:\Users\julien.mortier\conda-envs\harpy\lib\site-packages\anndata\_core\anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
<tifffile.TiffFile 'SCA003_HE_hires.tif'> ImageJ series metadata invalid or corrupted file


INFO     Transposing `data` of type: <class 'dask.array.core.Array'> to ('c', 'y', 'x').                           
INFO     The instance_key column in 'table.obs' ('table.obs[location_id]') will be relabeled to ensure a numeric   
         data type, with a continuous range and without including the value 0 (which is reserved for the           
         background). The new labels will be stored in a new column named 'relabeled_location_id'.                 


c:\Users\julien.mortier\conda-envs\harpy\lib\site-packages\spatialdata\_core\spatialdata.py:511: UserWarning: Converting `region_key: region` to categorical dtype.
  convert_region_column_to_categorical(table)


INFO     The instance_key column in 'table.obs' ('table.obs[location_id]') will be relabeled to ensure a numeric   
         data type, with a continuous range and without including the value 0 (which is reserved for the           
         background). The new labels will be stored in a new column named 'relabeled_location_id'.                 
INFO     The instance_key column in 'table.obs' ('table.obs[location_id]') will be relabeled to ensure a numeric   
         data type, with a continuous range and without including the value 0 (which is reserved for the           
         background). The new labels will be stored in a new column named 'relabeled_location_id'.                 
INFO     The instance_key column in 'table.obs' ('table.obs[location_id]') will be relabeled to ensure a numeric   
         data type, with a continuous range and without including the value 0 (which is reserved for the           
         background). The new labels will be stored in a new column name

c:\Users\julien.mortier\conda-envs\harpy\lib\site-packages\spatialdata\_core\spatialdata.py:511: UserWarning: Converting `region_key: region` to categorical dtype.
  convert_region_column_to_categorical(table)
c:\Users\julien.mortier\conda-envs\harpy\lib\site-packages\spatialdata\_core\spatialdata.py:511: UserWarning: Converting `region_key: region` to categorical dtype.
  convert_region_column_to_categorical(table)
c:\Users\julien.mortier\conda-envs\harpy\lib\site-packages\spatialdata\_core\spatialdata.py:511: UserWarning: Converting `region_key: region` to categorical dtype.
  convert_region_column_to_categorical(table)
<tifffile.TiffFile 'SCA003_HE_hires.tif'> ImageJ series metadata invalid or corrupted file


INFO     The Zarr backing store has been changed from None the new file path:                                      
         D:\2024-07-UTBenchmark-SpC\data\processed\Visium_HD\SCA003\harpy\sdata.zarr                               


version mismatch: detected: RasterFormatV02, requested: FormatV04
c:\Users\julien.mortier\conda-envs\harpy\lib\site-packages\zarr\creation.py:614: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)
version mismatch: detected: RasterFormatV02, requested: FormatV04
c:\Users\julien.mortier\conda-envs\harpy\lib\site-packages\zarr\creation.py:614: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)
c:\Users\julien.mortier\conda-envs\harpy\lib\site-packages\zarr\creation.py:614: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)
c:\Users\julien.mortier\conda-envs\harpy\lib\site-packages\zarr\creation.py:614: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)
c:\Users\julien.mortier\conda-envs\harpy\lib

SpatialData object, with associated Zarr store: D:\2024-07-UTBenchmark-SpC\data\processed\Visium_HD\SCA003\harpy\sdata.zarr
├── Images
│     ├── 'SCA003_cytassist_image': DataArray[cyx] (3, 3000, 3203)
│     ├── 'SCA003_full_image': DataTree[cyx] (3, 15241, 16021), (3, 7620, 8010), (3, 3810, 4005), (3, 1905, 2002), (3, 952, 1001)
│     ├── 'SCA003_hires_image': DataArray[cyx] (3, 5708, 6000)
│     └── 'SCA003_lowres_image': DataArray[cyx] (3, 571, 600)
├── Labels
│     ├── 'SCA003_square_002um_labels': DataArray[yx] (3350, 3350)
│     ├── 'SCA003_square_008um_labels': DataArray[yx] (838, 838)
│     ├── 'SCA003_square_016um_labels': DataArray[yx] (419, 419)
│     └── 'SCA003_square_032um_labels': DataArray[yx] (210, 210)
├── Shapes
│     ├── 'SCA003_square_002um': GeoDataFrame shape: (7520279, 1) (2D shapes)
│     ├── 'SCA003_square_008um': GeoDataFrame shape: (673473, 1) (2D shapes)
│     ├── 'SCA003_square_016um': GeoDataFrame shape: (175199, 1) (2D shapes)
│     └── 'SCA003_square_03

In [7]:
levels = [*sdata.tables]
print(f"{pipeline_name} levels: {levels}")

SpaceRanger levels: ['square_002um', 'square_008um', 'square_016um', 'square_032um']


## Specify plotting parameters

In [8]:
DPI = 200
MAX_DIM_FOR_PLOTS = 10000

HE_image = f'{dataset_id}_full_image'

best_scale = best_datatree_scale(sdata[HE_image], MAX_DIM_FOR_PLOTS)
print(f'best scale: {best_scale}')

figsize_factor = DPI
figsize = (sdata.images[HE_image][best_scale]['image'].shape[2]/figsize_factor, sdata.images[HE_image][best_scale]['image'].shape[1]/figsize_factor)
print(f'figsize: {figsize}')

plots_path = os.path.join(os.path.dirname(output_path), 'plots')
os.makedirs(plots_path, exist_ok=True)
print(f'plots_path: {plots_path}')

best scale: scale1
figsize: (40.05, 38.1)
plots_path: D:/2024-07-UTBenchmark-SpC/data/processed/Visium_HD/SCA003/harpy\plots


## Plot H&E

In [9]:
fig, ax = plt.subplots(figsize=figsize, dpi=DPI)

title = "H&E"

sdata.pl.render_images(
    element=HE_image, scale=best_scale
).pl.show(
    coordinate_systems=dataset_id,
    title=title,
    figsize=figsize,
    colorbar=False,
    dpi=DPI,
    ax=ax,
)

ax.set_title(title, fontsize=rel_fontsize(fig, 0.02))
ax.set_xlabel("x", fontsize=rel_fontsize(fig, 0.02))
ax.set_ylabel("y", fontsize=rel_fontsize(fig, 0.02))
ax.tick_params(axis="both", which="both", labelsize=rel_fontsize(fig, 0.015))

save_path = os.path.join(plots_path, f'HE.png')
fig.savefig(save_path, bbox_inches="tight", pad_inches=max(fig.get_size_inches())/100, facecolor="white", transparent=False)

plt.close(fig)

print("Plot saved at:", save_path)

Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-0.003937008..1.0].


Plot saved at: D:/2024-07-UTBenchmark-SpC/data/processed/Visium_HD/SCA003/harpy\plots\HE.png


## Plot Space Ranger levels and tissue annotation

In [ ]:
for level in [level for level in levels if level not in ['square_002um']]:
    gc.collect()
    fig, ax = plt.subplots(figsize=figsize, dpi=DPI)

    title = f"{pipeline_name} {level} - {pipeline_name} tissue annotation"
    
    sdata.pl.render_images(
        element=HE_image, scale=best_scale
    ).pl.render_labels(
        element=f"{dataset_id}_{level}_labels", fill_alpha=0.5, contour_px=5, outline_alpha=0, color='in_tissue', scale=best_scale
    ).pl.show(
        coordinate_systems=dataset_id,
        title=title,
        figsize=figsize,
        colorbar=False,
        dpi=DPI,
        ax=ax,
    )

    ax.set_title(title, fontsize=rel_fontsize(fig, 0.02))
    ax.set_xlabel("x", fontsize=rel_fontsize(fig, 0.02))
    ax.set_ylabel("y", fontsize=rel_fontsize(fig, 0.02))
    ax.tick_params(axis="both", which="both", labelsize=rel_fontsize(fig, 0.015))

    save_path = os.path.join(plots_path, f'{pipeline_name}_tissue_annotation_{level}.png')
    fig.savefig(save_path, bbox_inches="tight", pad_inches=max(fig.get_size_inches())/100, facecolor="white", transparent=False)

    plt.close(fig)

    print("Plot saved at:", save_path)

Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-0.003937008..1.0].


Plot saved at: D:/2024-07-UTBenchmark-SpC/data/processed/Visium_HD/SCA003/harpy\plots\SpaceRanger_tissue_annotation_square_008um.png


Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-0.003937008..1.0].


Plot saved at: D:/2024-07-UTBenchmark-SpC/data/processed/Visium_HD/SCA003/harpy\plots\SpaceRanger_tissue_annotation_square_016um.png


Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-0.003937008..1.0].


Plot saved at: D:/2024-07-UTBenchmark-SpC/data/processed/Visium_HD/SCA003/harpy\plots\SpaceRanger_tissue_annotation_square_032um.png


## Calculate pixel size

In [12]:
# Determine pixel size in microns
import tifffile

with tifffile.TiffFile(path_img) as tif:
    xres = tif.pages[0].tags["XResolution"].value  # (numerator, denominator)
    yres = tif.pages[0].tags["YResolution"].value
    unit = tif.pages[0].tags["ResolutionUnit"].value

    px_per_cm_x = xres[0] / xres[1]
    px_per_cm_y = yres[0] / yres[1]

    if unit.name == "CENTIMETER":
        microns_per_pixel_x = 10_000 / px_per_cm_x
        microns_per_pixel_y = 10_000 / px_per_cm_y
    elif unit.name == "INCH":
        microns_per_pixel_x = 25_400 / px_per_cm_x
        microns_per_pixel_y = 25_400 / px_per_cm_y
    else:
        raise ValueError(f"Unknown ResolutionUnit: {unit}")

if microns_per_pixel_x == microns_per_pixel_y:
    microns_per_pixel = microns_per_pixel_x
    print(f"Pixel size: {microns_per_pixel:.3f} µm")
else:
    print(f"Pixel size: {microns_per_pixel_x:.3f} µm (X), {microns_per_pixel_y:.3f} µm (Y)")


Pixel size: 0.442 µm


## Create bounding box for total capture area

In [12]:
# Add bounding box for total capture area
import shapely

geoms = sdata.shapes[f'{dataset_id}_square_002um'].geometry
bbox = shapely.MultiPolygon(list(geoms)).minimum_rotated_rectangle

sdata = hp.sh.add_shapes_layer(
    sdata,
    input = gpd.GeoDataFrame(geometry=[bbox]),
    output_layer = f'{dataset_id}_capture_area',
    transformations = get_transformation(sdata.shapes[f'{dataset_id}_square_002um'], get_all=True),
    overwrite = True,
)

In [13]:
# Export bbox GeoJSON
sdata.shapes[f'{dataset_id}_capture_area'].to_file(os.path.join(os.path.dirname(output_path), f'{dataset_id}_capture_area.geojson'), driver="GeoJSON")

c:\Users\julien.mortier\conda-envs\harpy\lib\site-packages\pyogrio\geopandas.py:710: UserWarning: 'crs' was not provided.  The output dataset will not have projection information defined and may not be usable in other systems.
  write(


In [14]:
# Check capture area bbox
fig, ax = plt.subplots(figsize=figsize, dpi=DPI)

title = "Capture area bounding box"

sdata.pl.render_images(
    element=HE_image, scale=best_scale
).pl.render_shapes(
    element=f'{dataset_id}_square_008um', color='black', fill_alpha=0.5, outline_width=rel_fontsize(fig, 0.0012), outline_alpha=1, outline_color='black'
).pl.render_shapes(
    element=f'{dataset_id}_capture_area', color='#1f77b4', fill_alpha=0.1, outline_width=rel_fontsize(fig, 0.002), outline_alpha=1, outline_color='red'
).pl.show(
    coordinate_systems=dataset_id,
    title=title,
    figsize=figsize,
    colorbar=False,
    dpi=DPI,
    ax=ax,
    pad_extent=rel_fontsize(fig, 0.03)
)

ax.set_title(title, fontsize=rel_fontsize(fig, 0.02))
ax.set_xlabel("x", fontsize=rel_fontsize(fig, 0.02))
ax.set_ylabel("y", fontsize=rel_fontsize(fig, 0.02))
ax.tick_params(axis="both", which="both", labelsize=rel_fontsize(fig, 0.015))

save_path = os.path.join(plots_path, f'capture_area_bbox.png')
fig.savefig(save_path, bbox_inches="tight", pad_inches=max(fig.get_size_inches())/100, facecolor="white", transparent=False)

plt.close(fig)

print("Plot saved at:", save_path)

INFO     Value for parameter 'color' appears to be a color, using it as such.                                      
INFO     Value for parameter 'color' appears to be a color, using it as such.                                      


Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-0.003937008..1.0].


INFO     Using 'datashader' backend with 'None' as reduction method to speed up plotting. Depending on the         
         reduction method, the value range of the plot might change. Set method to 'matplotlib' do disable this    
         behaviour.                                                                                                
Plot saved at: D:/2024-07-UTBenchmark-SpC/data/processed/Visium_HD/SCA003/harpy\plots\capture_area_bbox.png


In [15]:
# Area of capture area in mm²
gdf = sdata.shapes[f'{dataset_id}_capture_area']
capture_area = (gdf.geometry.area.sum() * microns_per_pixel**2) / 1000000
print(f"Area of capture area: {capture_area:.2f} mm²")

Area of capture area: 44.86 mm²


In [16]:
from math import dist
coords = list(bbox.exterior.coords)

# Compute lengths of bbox
side1 = dist(coords[0], coords[1]) * microns_per_pixel / 10000
side2 = dist(coords[1], coords[2]) * microns_per_pixel / 10000

print(f"Dimensions of capture area bbox: {side1:.2f} cm x {side2:.2f} cm")

Dimensions of capture area bbox: 0.67 cm x 0.67 cm


## Add tissue annotation

In [17]:
# Add tissue annotation as shapes layer and as labels layer
gdf = gpd.read_file(tissue_annotation_file)
gdf = gdf.explode(index_parts=False) # This avoid problems with MultiPolygon geometries
gdf.index = gdf.index + 1 # Index 0 is not allowed

sdata = hp.sh.add_shapes_layer(
    sdata,
    input = gdf,
    output_layer = f'{dataset_id}_tissue',
    transformations = get_transformation(sdata.shapes[f'{dataset_id}_square_002um'], get_all=True),
    overwrite = True,
)

sdata = hp.im.rasterize(
    sdata,
    shapes_layer = f'{dataset_id}_tissue',
    output_layer = f'{dataset_id}_tissue_mask',
    chunks = 1024,
    overwrite = True,
)

2025-08-26 11:08:59,144 - harpy.image._manager - INFO - Writing results to layer 'SCA003_tissue_mask'


In [18]:
# Check tissue annotation
fig, ax = plt.subplots(figsize=figsize, dpi=DPI)

title = "Tissue annotation"

sdata.pl.render_images(
    element=HE_image, scale=best_scale
).pl.render_shapes(
    element=f'{dataset_id}_tissue', color='#1f77b4', fill_alpha=0.3, outline_width=rel_fontsize(fig, 0.002), outline_alpha=1, outline_color='red'
).pl.show(
    coordinate_systems=dataset_id,
    title=title,
    figsize=figsize,
    colorbar=False,
    dpi=DPI,
    ax=ax,
    pad_extent=rel_fontsize(fig, 0.02)
)

ax.set_title(title, fontsize=rel_fontsize(fig, 0.02))
ax.set_xlabel("x", fontsize=rel_fontsize(fig, 0.02))
ax.set_ylabel("y", fontsize=rel_fontsize(fig, 0.02))
ax.tick_params(axis="both", which="both", labelsize=rel_fontsize(fig, 0.015))

save_path = os.path.join(plots_path, f'qupath_tissue_annotation.png')
fig.savefig(save_path, bbox_inches="tight", pad_inches=max(fig.get_size_inches())/100, facecolor="white", transparent=False)

plt.close(fig)

print("Plot saved at:", save_path)

INFO     Value for parameter 'color' appears to be a color, using it as such.                                      


Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-0.003937008..1.0].


Plot saved at: D:/2024-07-UTBenchmark-SpC/data/processed/Visium_HD/SCA003/harpy\plots\qupath_tissue_annotation.png


In [15]:
# Determine which spots are in tissue annotation
for level in levels:
    gc.collect()
    sdata = add_in_tissue(
        sdata, 
        table_layer = level, 
        tissue_mask = sdata.labels[f'{dataset_id}_tissue_mask'].compute().data
    )

c:\Users\julien.mortier\conda-envs\harpy\lib\site-packages\spatialdata\_core\spatialdata.py:184: UserWarning: The table is annotating 'SCA003_square_002um_labels', which is not present in the SpatialData object.
  self.validate_table_in_spatialdata(v)
c:\Users\julien.mortier\conda-envs\harpy\lib\site-packages\spatialdata\_core\spatialdata.py:184: UserWarning: The table is annotating 'SCA003_square_008um_labels', which is not present in the SpatialData object.
  self.validate_table_in_spatialdata(v)
c:\Users\julien.mortier\conda-envs\harpy\lib\site-packages\spatialdata\_core\spatialdata.py:184: UserWarning: The table is annotating 'SCA003_square_016um_labels', which is not present in the SpatialData object.
  self.validate_table_in_spatialdata(v)
c:\Users\julien.mortier\conda-envs\harpy\lib\site-packages\spatialdata\_core\spatialdata.py:184: UserWarning: The table is annotating 'SCA003_square_032um_labels', which is not present in the SpatialData object.
  self.validate_table_in_spatial

In [18]:
# Check spots in tissue
for level in [level for level in levels if level not in ['square_002um']]:
    gc.collect()
    fig, ax = plt.subplots(figsize=figsize, dpi=DPI)

    title = "Tissue annotation spots"

    sdata.pl.render_images(
        element=HE_image, scale=best_scale
    ).pl.render_labels(
        element=f"{dataset_id}_{level}_labels", fill_alpha=0.5, contour_px=5, outline_alpha=0, color='in_tissue_annotation', table_layer=level, scale=best_scale
    ).pl.show(
        coordinate_systems=dataset_id,
        title=title,
        figsize=figsize,
        colorbar=False,
        dpi=DPI,
        ax=ax,
        pad_extent=rel_fontsize(fig, 0.02)
    )

    ax.set_title(title, fontsize=rel_fontsize(fig, 0.02))
    ax.set_xlabel("x", fontsize=rel_fontsize(fig, 0.02))
    ax.set_ylabel("y", fontsize=rel_fontsize(fig, 0.02))
    ax.tick_params(axis="both", which="both", labelsize=rel_fontsize(fig, 0.015))

    save_path = os.path.join(plots_path, f'qupath_annotation_spots_{level}.png')
    fig.savefig(save_path, bbox_inches="tight", pad_inches=max(fig.get_size_inches())/100, facecolor="white", transparent=False)

    plt.close(fig)

    print("Plot saved at:", save_path)

Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-0.003937008..1.0].


Plot saved at: D:/2024-07-UTBenchmark-SpC/data/processed/Visium_HD/SCA003/harpy\plots\qupath_annotation_spots_square_008um.png


Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-0.003937008..1.0].


Plot saved at: D:/2024-07-UTBenchmark-SpC/data/processed/Visium_HD/SCA003/harpy\plots\qupath_annotation_spots_square_016um.png


c:\Users\julien.mortier\conda-envs\harpy\lib\site-packages\spatialdata_plot\pl\utils.py:1708: UserWarning: Multiple tables contain color column, using square_032um_e5ba960b-2a0b-45ce-8947-43b659f66152
  color, table_name = _validate_col_for_column_table(sdata, el, color, param_dict["table_name"], labels=True)
Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-0.003937008..1.0].


Plot saved at: D:/2024-07-UTBenchmark-SpC/data/processed/Visium_HD/SCA003/harpy\plots\qupath_annotation_spots_square_032um.png


In [7]:
tissue_positions = pd.read_parquet(os.path.join(path_outs, 'binned_outputs/square_002um/spatial/tissue_positions.parquet'))
tissue_positions

,barcode,in_tissue,array_row,array_col,pxl_row_in_fullres,pxl_col_in_fullres
0,s_002um_00000_00000-1,0,0,0,15228.925668,1700.316115
1,s_002um_00000_00001-1,0,0,1,15228.946621,1704.834104
2,s_002um_00000_00002-1,0,0,2,15228.967574,1709.352094
3,s_002um_00000_00003-1,0,0,3,15228.988527,1713.870083
4,s_002um_00000_00004-1,0,0,4,15229.009479,1718.388073
...,...,...,...,...,...,...
11222495,s_002um_03349_03345-1,0,3349,3345,158.229877,16876.306000
11222496,s_002um_03349_03346-1,0,3349,3346,158.250823,16880.824085
11222497,s_002um_03349_03347-1,0,3349,3347,158.271769,16885.342170
11222498,s_002um_03349_03348-1,0,3349,3348,158.292715,16889.860255


In [8]:
# Total number of spots in capture area
print(f"Total number of spots in capture area: {len(tissue_positions)}")

Total number of spots in capture area: 11222500


In [9]:
# Total number of spots in tissue
tissue_mask = sdata.labels[f'{dataset_id}_tissue_mask'].compute().data
mask_shape = tissue_mask.shape

tissue_positions["in_tissue_annotation"] = tissue_positions.apply(
    lambda row: assign_region((row["pxl_col_in_fullres"], row["pxl_row_in_fullres"]), tissue_mask, mask_shape), axis=1
)

total_n_spots_in_tissue = len(tissue_positions[tissue_positions['in_tissue_annotation'] == 1])

print(f"Total number of spots in tissue: {total_n_spots_in_tissue}")

Total number of spots in tissue: 6217015


In [10]:
# Total number of spots in table
n_spots = len(sdata.tables['square_002um'].obs)
print(f"Number of spots in table: {n_spots}")

Number of spots in table: 7520279


In [11]:
# Number of spots in tissue (in table)
n_spots_in_tissue = len(sdata.tables['square_002um'][sdata.tables['square_002um'].obs['in_tissue_annotation'] == 1].obs)
print(f"Number of spots in tissue: {n_spots_in_tissue}")

Number of spots in tissue: 6216979


In [12]:
# Save barcodes
with open(os.path.join(os.path.dirname(output_path), "all_barcodes.txt"), "w") as f:
    for item in tissue_positions['barcode']:
        f.write(f"{item}\n")
        
with open(os.path.join(os.path.dirname(output_path), "barcodes_in_tissue.txt"), "w") as f:
    for item in tissue_positions[tissue_positions['in_tissue_annotation'] == 1]['barcode']:
        f.write(f"{item}\n")      

In [13]:
# Area of tissue annotation that overlaps with capture area in mm²
tissue_gdf = sdata.shapes[f"{dataset_id}_tissue"].unary_union
capture_gdf = sdata.shapes[f"{dataset_id}_capture_area"].unary_union

overlap_geom = tissue_gdf.intersection(capture_gdf)

overlap_area_mm2 = (overlap_geom.area * microns_per_pixel**2) / 1000000

print(f"Capture area covered by tissue: {overlap_area_mm2:.2f} mm²")

Capture area covered by tissue: 24.84 mm²


## Capture effeciency metrics

In [9]:
probe_set_df = pd.read_csv(f"{path_outs}/probe_set.csv", comment='#')
included = set(probe_set_df[probe_set_df['included'] == True]['gene_id'])
adata = sdata.tables['square_002um'][:, sdata.tables['square_002um'].var['gene_ids'].isin(included)]

In [28]:
# Total number of spots in table with at least one UMI
n_spots_not_zero = len(adata.obs[adata.X.sum(axis=1) > 0])
print(f"Number of spots in table with at least one UMI: {n_spots_not_zero}")

Number of spots in table with at least one UMI: 7478864


In [29]:
# Number of spots in tissue with at least one UMI
in_tissue_mask = adata.obs['in_tissue_annotation'] == 1
umis_mask = np.array(adata.X.sum(axis=1)).flatten() > 0
spots_mask = in_tissue_mask & umis_mask
n_spots_in_tissue_not_zero = spots_mask.sum()
print(f"Number of spots in tissue with at least one UMI: {n_spots_in_tissue_not_zero}")

Number of spots in tissue with at least one UMI: 6200536


In [30]:
# Total number of genes targeted
print(f"Total number of genes targeted: {len(included)}")

Total number of genes targeted: 19059


In [31]:
# Total number of genes
print(f"Total number of genes: {(adata.X.sum(axis=0) > 0).sum()}")

Total number of genes: 19011


In [32]:
# Number of genes in tissue
print(f"Number of genes in tissue: {(adata[adata.obs['in_tissue'] == 1].X.sum(axis=0) > 0).sum()}")

Number of genes in tissue: 19011


In [33]:
# Total number of unique UMIs
print(f"Total number of unique UMIs: {adata.X.sum()}")

Total number of unique UMIs: 216805264.0


In [34]:
# Number of unique UMIs in tissue
print(f"Number of unique UMIs in tissue: {adata[adata.obs['in_tissue_annotation'] == 1].X.sum()}")

Number of unique UMIs in tissue: 213825104.0


In [35]:
import h5py

# Open the HDF5 file
with h5py.File(f"{path_outs}/molecule_info.h5", "r") as f:
    probe_idx = f['/probes/probe_id']
    probe_id = f['/probes/probe_id']
    count = f["/count"]

    total_sum = count[:].sum()
    total_count = count.size

    print("Sum of all numbers:", total_sum)
    print("Total number of entries:", total_count)


Sum of all numbers: 545172636
Total number of entries: 217075355


In [36]:
import h5py
import numpy as np

# Open the HDF5 file
with h5py.File(f"{path_outs}/molecule_info.h5", "r") as f:
    # Load the feature_id, feature_idx, and count datasets
    feature_id = f['/features/id'][:]
    feature_idx = f['/feature_idx'][:]
    count = f['/count'][:]

    # Decode feature_id entries
    feature_id = np.array([i.decode('ascii') for i in feature_id])
    
    # Find indices to keep
    included_feature_mask = np.isin(feature_id, list(included))
    included_feature_indices = np.where(included_feature_mask)[0]
    included_counts_mask = np.isin(feature_idx, included_feature_indices)
    
    # Sum counts for filtered indices
    included_counts = count[included_counts_mask]
    total_sum = np.sum(included_counts)
    total_count = len(included_counts)

    print("Total number of UMIs for included genes:", total_sum)
    print("Number of unique UMIs for included genes:", total_count)


Total number of UMIs for included genes: 544499100
Number of unique UMIs for included genes: 216805241


## Export unique UMI counts per gene

In [14]:
adata_in_tissue = adata[adata.obs['in_tissue_annotation'] == 1]

gene_counts = np.array(adata_in_tissue.X.sum(axis=0)).flatten()

df_gene_counts = pd.DataFrame({
    "gene": adata_in_tissue.var_names,
    "total_counts": gene_counts,
    "total_counts_corrected": gene_counts / overlap_area_mm2
})

df_gene_counts.to_csv(os.path.join(os.path.dirname(output_path), "gene_total_counts.csv"), index=False)

## QC

In [15]:
# Calculate QC metrics using scanpy's calculate_qc_metrics()
for level in levels:
    gc.collect()
    sdata = qc_metrics(
        sdata,
        level,
        species
    )

c:\Users\julien.mortier\conda-envs\harpy\lib\site-packages\spatialdata\_core\spatialdata.py:184: UserWarning: The table is annotating 'SCA003_square_002um_labels', which is not present in the SpatialData object.
  self.validate_table_in_spatialdata(v)
c:\Users\julien.mortier\conda-envs\harpy\lib\site-packages\spatialdata\_core\spatialdata.py:184: UserWarning: The table is annotating 'SCA003_square_008um_labels', which is not present in the SpatialData object.
  self.validate_table_in_spatialdata(v)
c:\Users\julien.mortier\conda-envs\harpy\lib\site-packages\spatialdata\_core\spatialdata.py:184: UserWarning: The table is annotating 'SCA003_square_016um_labels', which is not present in the SpatialData object.
  self.validate_table_in_spatialdata(v)
c:\Users\julien.mortier\conda-envs\harpy\lib\site-packages\spatialdata\_core\spatialdata.py:184: UserWarning: The table is annotating 'SCA003_square_032um_labels', which is not present in the SpatialData object.
  self.validate_table_in_spatial

In [16]:
# Plot QC metrics
for level in [level for level in levels if level not in ['square_002um']]:
    gc.collect()
    for qc_metric in ['n_genes_by_counts', 'total_counts', 'pct_counts_in_top_50_genes', 'total_counts_mt', 'pct_counts_mt']:
        gc.collect()
        values = sdata.tables[level].obs[qc_metric].to_numpy()
        vmin, vmax = np.percentile(values, [1, 99])
        norm = matplotlib.colors.Normalize(vmin=vmin, vmax=vmax)
        
        fig, ax = plt.subplots(figsize=figsize, dpi=DPI)

        title = f"scanpy - {qc_metric}"

        sdata.pl.render_images(
            element=HE_image, scale=best_scale
        ).pl.render_labels(
            element=f"{dataset_id}_{level}_labels", fill_alpha=1, contour_px=5, outline_alpha=0, color=qc_metric, scale=best_scale, table_name=level, cmap="viridis", norm=norm
        ).pl.show(
            coordinate_systems=dataset_id,
            title=title,
            figsize=figsize,
            colorbar=True,
            dpi=DPI,
            ax=ax,
        )

        ax.set_title(title, fontsize=rel_fontsize(fig, 0.02))
        ax.set_xlabel("x", fontsize=rel_fontsize(fig, 0.02))
        ax.set_ylabel("y", fontsize=rel_fontsize(fig, 0.02))
        ax.tick_params(axis="both", which="both", labelsize=rel_fontsize(fig, 0.015))

        cbar = fig.axes[-1]
        cbar.set_ylabel(qc_metric, rotation=270, labelpad=rel_fontsize(fig, 0.025), fontsize = rel_fontsize(fig, 0.02))
        cbar.tick_params(axis="y", which="both", labelsize=rel_fontsize(fig, 0.015))

        save_path = os.path.join(plots_path, f'scanpy_{qc_metric}_{level}.png')
        fig.savefig(save_path, bbox_inches="tight", pad_inches=max(fig.get_size_inches())/100, facecolor="white", transparent=False)

        plt.close(fig)

        print("Plot saved at:", save_path)

Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-0.003937008..1.0].


Plot saved at: D:/2024-07-UTBenchmark-SpC/data/processed/Visium_HD/SCA003/harpy\plots\scanpy_n_genes_by_counts_square_008um.png


Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-0.003937008..1.0].


Plot saved at: D:/2024-07-UTBenchmark-SpC/data/processed/Visium_HD/SCA003/harpy\plots\scanpy_total_counts_square_008um.png


Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-0.003937008..1.0].


...

Plot saved at: D:/2024-07-UTBenchmark-SpC/data/processed/Visium_HD/SCA003/harpy\plots\scanpy_pct_counts_in_top_50_genes_square_032um.png


Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-0.003937008..1.0].


Plot saved at: D:/2024-07-UTBenchmark-SpC/data/processed/Visium_HD/SCA003/harpy\plots\scanpy_total_counts_mt_square_032um.png


Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-0.003937008..1.0].


Plot saved at: D:/2024-07-UTBenchmark-SpC/data/processed/Visium_HD/SCA003/harpy\plots\scanpy_pct_counts_mt_square_032um.png
